# KLTN — V1.4: StratifiedGroupKFold + Patience cao (khắc phục outlier seed 999)

**Vấn đề V1.3 multi-seed:** Mean F1 = 0,6765 ± 0,1714. Seed 999 sụp xuống 0,3749 (dừng sớm ở epoch 39).

**Hai giả thuyết được kiểm chứng ở V1.4:**

1. **Class imbalance ở test set** → dùng `StratifiedGroupKFold` (cân bằng cả class lẫn clip)
2. **Early stop quá sớm với seed khó** → tăng patience 20 → 35, hoặc bỏ early stop hoàn toàn (chạy full epoch)

**Mọi thứ khác giữ giống V1.3:** BiLSTM, Dropout, Focal Loss, AdamW, Gaussian noise + Time crop.

**Mong đợi:** Std giảm xuống ≤ 0,03 và mean ≥ 0,78 → ổn định, qua ngưỡng đề cương.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
ROOT_DIR = "/content/drive/MyDrive/KLTN/Human-Reco"   # ★ Sửa nếu khác
OUT_DIR = f"{ROOT_DIR}/training_outputs/v1.4_stratified"
os.makedirs(OUT_DIR, exist_ok=True)
print(f"OUT_DIR = {OUT_DIR}")

In [ ]:
!pip install -q tensorflow scikit-learn pandas seaborn matplotlib numpy --upgrade
import sklearn
print("sklearn:", sklearn.__version__)  # cần >= 1.0 để có StratifiedGroupKFold

In [ ]:
import os, json, time, glob, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (LSTM, Bidirectional, Conv2D, MaxPooling2D,
                                     GlobalAveragePooling2D, BatchNormalization,
                                     Dense, Reshape, Input, Dropout)
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score, accuracy_score
print("TF:", tf.__version__, "GPU:", tf.config.list_physical_devices('GPU'))

## 2. Load + regenerate 56-feat data

In [ ]:
def regenerate_56feat(processed_root, seq_length=90, step=30):
    X, y, cids = [], [], []
    for lbl in ['normal', 'shoplifting']:
        v = 0 if lbl == 'normal' else 1
        for fp in sorted(glob.glob(os.path.join(processed_root, lbl, "*.csv"))):
            cid = f"{lbl}/{os.path.basename(fp)}"
            d = pd.read_csv(fp).values
            if d.shape[1] != 56: continue
            for i in range(0, len(d) - seq_length + 1, step):
                X.append(d[i: i + seq_length]); y.append(v); cids.append(cid)
    return np.array(X, dtype='float32'), np.array(y, dtype='int'), np.array(cids)

X, y, clip_ids = regenerate_56feat(f"{ROOT_DIR}/processed_data1")
print(f"X={X.shape}, y={y.shape}, clips={len(np.unique(clip_ids))}")

## 3. Components giống V1.3 (augmentation + focal loss + model)

In [ ]:
def augment_sample(s, noise_std=0.02, crop_prob=0.3, training=True):
    if not training: return s
    s = s.copy()
    if np.random.rand() < 0.8:
        s = s + np.random.normal(0, noise_std, s.shape).astype(np.float32)
    if np.random.rand() < crop_prob:
        T = s.shape[0]; c = np.random.randint(60, T + 1)
        if c < T:
            st = np.random.randint(0, T - c + 1)
            cr = s[st: st + c]
            i_src = np.linspace(0, c - 1, num=T)
            ifl = np.floor(i_src).astype(int); ic = np.minimum(ifl + 1, c - 1)
            w = (i_src - ifl).astype(np.float32)[:, None]
            s = (1 - w) * cr[ifl] + w * cr[ic]
    return s.astype(np.float32)


class AugGen(Sequence):
    def __init__(self, X, y, bs=64, augment=True, shuffle=True):
        self.X, self.y, self.bs = X, y, bs
        self.augment = augment; self.shuffle = shuffle
        self.idx = np.arange(len(X))
        if shuffle: np.random.shuffle(self.idx)
    def __len__(self): return int(np.ceil(len(self.X) / self.bs))
    def __getitem__(self, i):
        sl = self.idx[i*self.bs:(i+1)*self.bs]
        return (np.stack([augment_sample(self.X[k], training=self.augment) for k in sl]),
                self.y[sl])
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.idx)


def focal_loss(alpha=0.25, gamma=2.0):
    def f(y_true, y_pred):
        y_true = tf.cast(y_true, tf.int32)
        oh = tf.cast(tf.one_hot(y_true, depth=tf.shape(y_pred)[-1]), y_pred.dtype)
        p = tf.clip_by_value(y_pred, 1e-8, 1 - 1e-8)
        ce = -oh * tf.math.log(p)
        return tf.reduce_mean(tf.reduce_sum(alpha * tf.math.pow(1 - p, gamma) * ce, axis=-1))
    return f


def build_v14():
    m = Sequential([
        Input(shape=(90, 56)),
        Bidirectional(LSTM(32, return_sequences=True)), Dropout(0.3),
        Bidirectional(LSTM(32, return_sequences=True)), Dropout(0.3),
        Reshape((90, 64, 1)),
        Conv2D(64, (5, 5), strides=(2, 2), padding='same', activation='relu'),
        MaxPooling2D((2, 2), strides=(2, 2)), Dropout(0.3),
        Conv2D(128, (3, 3), strides=(1, 1), padding='same', activation='relu'),
        GlobalAveragePooling2D(), BatchNormalization(), Dropout(0.5),
        Dense(2, activation='softmax'),
    ])
    m.compile(optimizer=AdamW(learning_rate=1e-3, weight_decay=5e-4),
              loss=focal_loss(0.25, 2.0), metrics=['accuracy'])
    return m

## 4. Stratified split-by-clip — đảm bảo cân bằng cả class lẫn clip

`StratifiedGroupKFold` (sklearn ≥ 1.0) chia dữ liệu thành K fold, đảm bảo:
- Mỗi clip chỉ ở 1 fold (group-aware)
- Tỉ lệ class trong mỗi fold gần với tỉ lệ chung (stratify)

Để có 3 tập train/val/test 70/15/15, ta dùng K=5 (mỗi fold ~20%): chọn 1 fold làm test, 1 fold làm val, 3 fold làm train (≈ 60% — hơi thấp hơn 70% nhưng cân bằng tốt).

In [ ]:
# Sanity check: tỉ lệ class theo clip
clip_label_map = {cid: y[clip_ids == cid][0] for cid in np.unique(clip_ids)}
n_normal_clips = sum(1 for v in clip_label_map.values() if v == 0)
n_shop_clips   = sum(1 for v in clip_label_map.values() if v == 1)
print(f"Số clip: Normal={n_normal_clips}, Shoplifting={n_shop_clips}")

# Để StratifiedGroupKFold làm việc đúng, mỗi clip phải có 1 nhãn duy nhất
# (đúng với data của chúng ta vì 1 clip = 1 hành vi).

def split_3way_stratified(X, y, groups, seed):
    """Chia 3 tập train/val/test theo StratifiedGroupKFold với 5 fold."""
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    folds = list(sgkf.split(X, y, groups))
    test_fold  = seed % 5             # fold index for test
    val_fold   = (seed + 1) % 5       # fold index for val
    idx_test = folds[test_fold][1]
    idx_val  = folds[val_fold][1]
    train_mask = np.ones(len(X), dtype=bool)
    train_mask[idx_test] = False
    train_mask[idx_val] = False
    idx_train = np.where(train_mask)[0]
    return idx_train, idx_val, idx_test


# Test với seed 999 — xem có giải quyết được vấn đề không
idx_tr, idx_v, idx_t = split_3way_stratified(X, y, clip_ids, 999)
print(f"\n[seed 999] Train={len(idx_tr)} ({100*len(idx_tr)/len(X):.0f}%), "
      f"Val={len(idx_v)} ({100*len(idx_v)/len(X):.0f}%), "
      f"Test={len(idx_t)} ({100*len(idx_t)/len(X):.0f}%)")
u, c = np.unique(y[idx_t], return_counts=True)
print(f"  Test class dist: {dict(zip(u.tolist(), c.tolist()))}")

## 5. Hàm train_one_seed_v14 — patience 35 thay vì 20

Các thay đổi so V1.3:
- `StratifiedGroupKFold` thay `GroupShuffleSplit`
- `patience=35` (tăng từ 20) — cho seed khó cơ hội converge
- `min_delta=1e-4` — không dừng khi cải thiện nhỏ

In [ ]:
def train_one_seed_v14(X, y, clip_ids, seed, out_dir, epochs=150, verbose=2):
    print(f"\n{'='*60}\nSEED = {seed}\n{'='*60}")
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

    idx_tr, idx_v, idx_t = split_3way_stratified(X, y, clip_ids, seed)
    X_tr, y_tr = X[idx_tr], y[idx_tr]
    X_v, y_v   = X[idx_v], y[idx_v]
    X_t, y_t   = X[idx_t], y[idx_t]

    # Assert no clip leak
    assert not (set(clip_ids[idx_tr]) & set(clip_ids[idx_v]))
    assert not (set(clip_ids[idx_tr]) & set(clip_ids[idx_t]))
    assert not (set(clip_ids[idx_v])  & set(clip_ids[idx_t]))

    # Class balance check
    u_test, c_test = np.unique(y_t, return_counts=True)
    print(f"Train={len(idx_tr)} | Val={len(idx_v)} | Test={len(idx_t)}, "
          f"test class dist: {dict(zip(u_test.tolist(), c_test.tolist()))}")

    train_gen = AugGen(X_tr, y_tr, bs=64, augment=True, shuffle=True)
    val_gen   = AugGen(X_v,  y_v,  bs=64, augment=False, shuffle=False)

    model = build_v14()
    ckpt = f"{out_dir}/best_seed{seed}.keras"
    cbs = [
        ModelCheckpoint(ckpt, monitor='val_loss', save_best_only=True),
        # patience cao + min_delta để tránh dừng sớm
        EarlyStopping(monitor='val_loss', patience=35, min_delta=1e-4,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10,
                          min_lr=1e-6, verbose=0),
    ]
    t0 = time.time()
    hist = model.fit(train_gen, validation_data=val_gen, epochs=epochs,
                     callbacks=cbs, verbose=verbose)
    train_t = time.time() - t0

    y_pred = model.predict(X_t, verbose=0).argmax(1)
    f1m = f1_score(y_t, y_pred, average='macro')
    f1c = f1_score(y_t, y_pred, average=None, zero_division=0)
    cm = confusion_matrix(y_t, y_pred)
    acc = accuracy_score(y_t, y_pred)
    prec = precision_score(y_t, y_pred, average='macro', zero_division=0)
    rec = recall_score(y_t, y_pred, average='macro', zero_division=0)

    log = {
        "seed": seed, "epochs_trained": len(hist.history['loss']),
        "train_time_sec": train_t,
        "train_size": int(len(idx_tr)), "val_size": int(len(idx_v)), "test_size": int(len(idx_t)),
        "test_f1_macro": float(f1m),
        "test_f1_per_class": [float(v) for v in f1c],
        "test_accuracy": float(acc),
        "test_precision_macro": float(prec),
        "test_recall_macro": float(rec),
        "test_confusion_matrix": cm.tolist(),
    }
    with open(f"{out_dir}/log_seed{seed}.json", "w") as f:
        json.dump(log, f, indent=2, ensure_ascii=False)
    print(f"\n[seed {seed}] F1={f1m:.4f}, Acc={acc:.4f}, epochs={len(hist.history['loss'])}")
    return log

## 6. Chạy 5 seed (~40-50 phút)

In [ ]:
SEEDS = [42, 123, 7, 2024, 999]
results = []
t0 = time.time()
for s in SEEDS:
    log = train_one_seed_v14(X, y, clip_ids, s, OUT_DIR, epochs=150, verbose=0)
    results.append(log)
    print(f"  → Done seed {s}, tổng {(time.time()-t0)/60:.1f} phút")
print(f"\nTổng: {(time.time()-t0)/60:.1f} phút")

## 7. Phân tích thống kê + so sánh V1.3 vs V1.4

In [ ]:
import numpy as np

def stat(vals):
    return {"mean": float(np.mean(vals)), "std": float(np.std(vals, ddof=1)),
            "min": float(np.min(vals)), "max": float(np.max(vals))}

f1s   = [r["test_f1_macro"] for r in results]
accs  = [r["test_accuracy"] for r in results]
precs = [r["test_precision_macro"] for r in results]
recs  = [r["test_recall_macro"] for r in results]

summary = {
    "experiment": "V1.4 multi-seed StratifiedGroupKFold + patience 35",
    "seeds": SEEDS,
    "f1_macro": stat(f1s), "accuracy": stat(accs),
    "precision_macro": stat(precs), "recall_macro": stat(recs),
    "individual_runs": [
        {"seed": r["seed"], "f1": r["test_f1_macro"], "acc": r["test_accuracy"],
         "epochs": r["epochs_trained"]} for r in results
    ],
}

print("="*70)
print("V1.4 MULTI-SEED SUMMARY")
print("="*70)
print(f"{'Metric':<22} {'Mean':>10} {'Std':>8} {'Min':>10} {'Max':>10}")
print("-"*70)
for k, v in [("F1-macro", summary["f1_macro"]),
             ("Accuracy", summary["accuracy"]),
             ("Precision", summary["precision_macro"]),
             ("Recall", summary["recall_macro"])]:
    print(f"{k:<22} {v['mean']:>10.4f} {v['std']:>8.4f} {v['min']:>10.4f} {v['max']:>10.4f}")
print("\nPer-seed F1:")
for r in results:
    print(f"  seed {r['seed']:>4d}: F1 = {r['test_f1_macro']:.4f}, epochs = {r['epochs_trained']}")

# So sánh V1.3 cũ
print("\n=== So với V1.3 (GroupShuffleSplit, patience 20) ===")
print(f"V1.3: mean=0.6765, std=0.1714 (seed 999 sụp F1=0.37)")
print(f"V1.4: mean={summary['f1_macro']['mean']:.4f}, std={summary['f1_macro']['std']:.4f}")
print(f"     Δ mean: {summary['f1_macro']['mean'] - 0.6765:+.4f}")
print(f"     Δ std:  {summary['f1_macro']['std'] - 0.1714:+.4f}")

with open(f"{OUT_DIR}/V1.4_multiseed_summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(f"\n✓ Saved {OUT_DIR}/V1.4_multiseed_summary.json")

In [ ]:
# Plot so sánh V1.3 vs V1.4
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

v13 = {42: 0.7455, 123: 0.7732, 7: 0.7845, 2024: 0.7047, 999: 0.3749}
v14 = {r["seed"]: r["test_f1_macro"] for r in results}

x = np.arange(len(SEEDS))
w = 0.35
axes[0].bar(x - w/2, [v13[s] for s in SEEDS], w, label='V1.3 GroupShuffleSplit',
            color='lightcoral', edgecolor='black')
axes[0].bar(x + w/2, [v14[s] for s in SEEDS], w, label='V1.4 StratifiedGroupKFold',
            color='steelblue', edgecolor='black')
axes[0].axhline(y=0.78, color='red', linestyle='--', alpha=0.5, label='Ngưỡng 0,78')
axes[0].set_xticks(x); axes[0].set_xticklabels([f"seed={s}" for s in SEEDS])
axes[0].set_ylabel('F1-macro'); axes[0].set_title('V1.3 vs V1.4 — F1 qua 5 seed')
axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim([0.3, 1.0])

# Box plot
axes[1].boxplot([list(v13.values()), list(v14.values())],
                labels=['V1.3', 'V1.4'], showmeans=True)
axes[1].axhline(y=0.78, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Phân phối F1 — V1.3 vs V1.4')
axes[1].grid(axis='y', alpha=0.3)
axes[1].set_ylim([0.3, 1.0])

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/v1.3_vs_v1.4.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Saved {OUT_DIR}/v1.3_vs_v1.4.png")

## 8. Quyết định bước kế tiếp

Gửi tôi `V1.4_multiseed_summary.json`. Tôi sẽ cập nhật báo cáo và đề xuất:

| Mean V1.4 | Std V1.4 | Quyết định |
|---:|---:|---|
| ≥ 0,80 | ≤ 0,03 | V1.4 mạnh và stable; làm V2.0 ST-GCN cho đóng góp học thuật |
| 0,78 - 0,79 | ≤ 0,03 | V1.4 đạt ngưỡng + stable; V2.0 để vượt 0,85 |
| 0,75 - 0,77 | ≤ 0,05 | Stable hơn V1.3 nhưng dưới ngưỡng; tiến V2.0 với hi vọng cao |
| Std > 0,05 | — | Vẫn không stable; vấn đề là dataset (155 clip quá nhỏ). Cần thu thêm data hoặc V2.0 + augment mạnh hơn |